# Delivery 2 · Silver
Typed and checked. The **quality gate** runs every rule first; if one fails, nothing is written and silver keeps the previous version. Checks against other domains read their silver tables through shortcuts.

In [ ]:
# Domain: delivery · owner: Project planning
# Attach this notebook to the lakehouse "Delivery" in the workspace "Solvi Delivery".
DOMAIN = "delivery"
LANDING_PATH = f"Files/landing/{DOMAIN}"   # upload this domain's CSV files here
TABLE_FORMAT = "delta"
LOCAL_PREFIX = ""                            # local testing sets this to the domain name
AS_OF = "2026-10-31"                         # demo data ends here; use the current date with live data

from pyspark.sql import functions as F, Window as W

def write_table(df, name):
    (df.write.format(TABLE_FORMAT).mode("overwrite").option("overwriteSchema", "true").saveAsTable(LOCAL_PREFIX + name))
    print(f"wrote {name:<30} {df.count():>9,} rows")

def read_table(name):                         # a table this domain owns
    return spark.read.table(LOCAL_PREFIX + name)

def read_shared(domain, name):                # another domain's table, through a OneLake shortcut named domain + name
    return spark.read.table(f"{domain}{name}")

In [ ]:
SCHEMA = {
 "crew": {
  "crew_id": "string",
  "crew_name": "string",
  "crew_type": "string",
  "supplier_id": "string"
 },
 "certification": {
  "crew_id": "string",
  "certification": "string",
  "valid_from": "date",
  "valid_to": "date"
 },
 "job": {
  "job_id": "string",
  "order_id": "string",
  "installation_id": "string",
  "delivery_id": "string",
  "crew_id": "string",
  "planned_date": "date",
  "job_date": "date",
  "status": "string",
  "location": "string",
  "ventilation_check_done": "int",
  "sja_done": "int",
  "battery_temp_c": "double",
  "acclimatised": "int",
  "commissioning_complete": "int"
 }
}
KEYS = {"crew": ["crew_id"], "certification": ["crew_id", "valid_from"], "job": ["job_id"]}
FKS = [["crew", "supplier_id", "supplier", "supplier_id", "supply"], ["certification", "crew_id", "crew", "crew_id", "delivery"], ["job", "crew_id", "crew", "crew_id", "delivery"], ["job", "delivery_id", "battery_delivery", "delivery_id", "supply"], ["job", "order_id", "sales_order", "order_id", "sales"]]      # child, column, parent, column, parent's domain
RANGES = [["job", "status IN ('Done','Planned')"], ["job", "status = 'Planned' OR installation_id IS NOT NULL"], ["certification", "valid_to > valid_from"]]

def clean(name):
    df = read_table(f"bronze{name.replace('_', '')}")
    return df.select(*[F.nullif(F.trim(F.col(c)), F.lit("")).cast(t).alias(c) for c, t in SCHEMA[name].items()])
silver = {n: clean(n) for n in SCHEMA}
def parent(table, dom):
    return silver[table] if dom == DOMAIN else read_shared(dom, f"silver{table.replace('_', '')}")

results = []
for t, keys in KEYS.items():
    df = silver[t]
    results += [(t, "key not null", df.filter(" OR ".join(f"{k} IS NULL" for k in keys)).count()),
                (t, "key unique", df.groupBy(*keys).count().filter("count > 1").count())]
for c, cc, p, pc, dom in FKS:
    orphans = silver[c].filter(F.col(cc).isNotNull()).join(parent(p, dom).select(F.col(pc).alias(cc)), cc, "left_anti").count()
    results.append((c, f"{cc} -> {dom}.{p}", orphans))
for t, cond in RANGES:
    results.append((t, cond, silver[t].filter(f"NOT ({cond})").count()))
dq = (spark.createDataFrame(results, "table string, rule string, failing_rows long")
        .withColumn("passed", F.col("failing_rows") == 0).withColumn("checked_at", F.current_timestamp()))
display(dq.orderBy("passed", "table"))
failed = dq.filter("NOT passed").count()
if failed:
    raise Exception(f"Quality gate failed in {DOMAIN}: {failed} rule(s). Nothing written to silver.")

In [ ]:
for name, df in silver.items():
    write_table(df, f"silver{name.replace('_', '')}")
write_table(dq, "silverdqlog")